# Chapter 3 — The Autograd Mental Model (Practice)

Work through these exercises **after reading** `notes/ch03-autograd-mental-model.md`.

Each exercise states the *decision you're practicing*, gives a stub cell to fill in, and is followed by a pre-written **verification cell** — run it to grade yourself. Don't peek at `solutions/` until the verification passes or you're genuinely stuck.

In [ ]:
# ============================================================
# TOPIC: Autograd — the dynamic graph, backward, and how to control both
# MATH:  chain rule: dL/dw = dL/dy_pred * dy_pred/dw
# REF:   B00 ch03 notes — autograd-mental-model
# ============================================================

# --- Imports ---
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Reproducibility & device ---
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version : {torch.__version__}")
print(f"device        : {device}  (every exercise here runs fine on CPU)")

## Exercise 1 — Chain Rule vs Autograd

One data point of linear regression: $\hat{y} = w \cdot x + b$, $\mathcal{L} = (\hat{y} - y)^2$, with $x=2$, $y=7$, $w=1.5$, $b=0.5$ (the exact numbers from notes §2).

Compute the forward pass and **both analytic gradients by the chain rule as plain Python floats**, then call `loss.backward()` and watch autograd produce the identical numbers.

**Decision you're practicing:** trusting — because you've verified — that `.grad` is nothing but the chain rule.

In [ ]:
input_x, target_y = 2.0, 7.0
w = torch.tensor(1.5, requires_grad=True)
b = torch.tensor(0.5, requires_grad=True)

# TODO forward: y_pred = w*x + b, loss = (y_pred - target)^2
loss = None

# TODO analytic chain rule, as plain floats:
#   dL/dy_pred = 2*(y_pred - y);   dL/dw = dL/dy_pred * x;   dL/db = dL/dy_pred * 1
analytic_grad_w = None
analytic_grad_b = None

# TODO: run backward

**Verification**

In [ ]:
# --- Verification: Exercise 1 ---
assert loss is not None, "fill in the stub above first"
assert abs(loss.item() - 12.25) < 1e-6, f"loss should be 12.25, got {loss.item()}"
assert w.grad is not None, "did you call loss.backward()?"
assert abs(analytic_grad_w - (-14.0)) < 1e-6, f"analytic dL/dw should be -14.0, got {analytic_grad_w}"
assert abs(analytic_grad_b - (-7.0)) < 1e-6,  f"analytic dL/db should be -7.0, got {analytic_grad_b}"
assert abs(w.grad.item() - analytic_grad_w) < 1e-6, "autograd and your chain rule disagree on w"
assert abs(b.grad.item() - analytic_grad_b) < 1e-6, "autograd and your chain rule disagree on b"
print("chain rule and autograd agree to 6 decimals ✓")
print("Exercise 1 passed ✓")

## Exercise 2 — Leaf Detective

Eight tensors, three properties each: predict `(is_leaf, requires_grad, has_grad_fn)` for every one **before running anything**. The rules from notes §1: you-created-it → leaf; op-result → non-leaf with `grad_fn`; requires_grad is infectious; `no_grad` and `detach` break the chain.

**Decision you're practicing:** reading a tensor's autograd status from *how it was made* — the skill behind debugging "why is .grad None?".

In [ ]:
plain         = torch.ones(3)
param_like    = torch.ones(3, requires_grad=True)
computed      = param_like * 2
detached      = computed.detach()
moved         = param_like.to(torch.float64)     # careful: is a dtype move an op?
sliced        = param_like[:2]
module_weight = nn.Linear(2, 2).weight
with torch.no_grad():
    under_no_grad = param_like * 2

# TODO: replace each None triple with (is_leaf, requires_grad, has_grad_fn) — three bools
predictions = {
    "plain":         None,
    "param_like":    None,
    "computed":      None,
    "detached":      None,
    "moved":         None,
    "sliced":        None,
    "module_weight": None,
    "under_no_grad": None,
}

**Verification**

In [ ]:
# --- Verification: Exercise 2 ---
tensors = {
    "plain": plain, "param_like": param_like, "computed": computed, "detached": detached,
    "moved": moved, "sliced": sliced, "module_weight": module_weight, "under_no_grad": under_no_grad,
}
explanations = {
    "plain":         "created by you, no grad requested → plain leaf",
    "param_like":    "created by you WITH requires_grad → requiring leaf (a hand-made parameter)",
    "computed":      "result of an op on a requiring tensor → non-leaf, carries MulBackward",
    "detached":      "detach() cuts it from the graph → new leaf, requires nothing",
    "moved":         ".to(dtype) is an operation (ToCopyBackward) → NON-leaf, still requires grad",
    "sliced":        "slicing is an op (SliceBackward) → non-leaf",
    "module_weight": "nn.Parameter — the canonical requiring leaf",
    "under_no_grad": "computed while the recorder was off → orphan leaf, requires nothing",
}
wrong = 0
for name, tensor in tensors.items():
    truth = (tensor.is_leaf, tensor.requires_grad, tensor.grad_fn is not None)
    assert predictions[name] is not None, f"no prediction for {name!r}"
    mark = "✓" if tuple(predictions[name]) == truth else "✗"
    wrong += tuple(predictions[name]) != truth
    print(f"{mark} {name:14s} truth={str(truth):21s} — {explanations[name]}")
assert wrong == 0, f"{wrong} prediction(s) wrong — the misses above show which rule to revisit"
print("Exercise 2 passed ✓")

## Exercise 3 — The Doubling Gradient

The buggy loop below has no `zero_grad`, so every `backward()` piles onto stale gradients. It fits `w` toward 3 on a convex one-parameter problem — watch it **overshoot past the optimum** as the stale pile grows.

Write `fixed_training` (identical, plus the one missing line) returning `(final_weight, grad_history)`.

**Decision you're practicing:** accumulation-as-bug — recognizing the symptom (gradients that keep growing / updates that overshoot) and where `zero_grad` belongs.

In [ ]:
def buggy_training(num_steps=5):
    weight = torch.tensor(0.0, requires_grad=True)
    optimizer = torch.optim.SGD([weight], lr=0.1)
    grad_history = []
    for step in range(num_steps):
        loss = (weight * 1.0 - 3.0) ** 2          # convex; optimum at w = 3
        loss.backward()                           # ← accumulates onto stale grads!
        grad_history.append(weight.grad.item())
        optimizer.step()
        print(f"  step {step}: w.grad = {weight.grad.item():+8.3f}   w = {weight.item():+.4f}")
    return weight.item(), grad_history

print("BUGGY (no zero_grad):")
buggy_final, buggy_grads = buggy_training()

# TODO: same loop, fixed. Return (final_weight, grad_history).
def fixed_training(num_steps=5):
    pass

**Verification**

In [ ]:
# --- Verification: Exercise 3 ---
result = fixed_training(num_steps=5)
assert result is not None, "fill in the stub above first"
fixed_final, fixed_grads = result

# step 0 from w=0: dL/dw = 2*(0-3) = -6 exactly — a fresh gradient, no stale pile
assert abs(fixed_grads[0] - (-6.0)) < 1e-6, f"first-step grad should be -6.0, got {fixed_grads[0]}"

# convex problem + correct grads → |grad| strictly shrinks every step
for step in range(1, len(fixed_grads)):
    assert abs(fixed_grads[step]) < abs(fixed_grads[step - 1]), \
        f"|grad| grew at step {step}: is zero_grad missing or misplaced?"

# the buggy run's step-1 grad is the fresh grad PLUS the stale -6
fresh_step1 = fixed_grads[1]
assert abs(buggy_grads[1] - (fresh_step1 + buggy_grads[0])) < 1e-6, "sanity: buggy grad = fresh + stale"

assert abs(fixed_final - 3.0) < abs(buggy_final - 3.0), "the fix should land closer to the optimum w=3"
print(f"fixed final w = {fixed_final:.4f} (optimum 3.0); buggy final w = {buggy_final:.4f} — overshot")
print("Exercise 3 passed ✓")

## Exercise 4 — Gradient Accumulation Done Right

`full_batch_step` (given) does one SGD step on the whole 4-sample batch. Implement `accumulation_step`: the **same single step** built from 2 micro-batches of 2 — backward each micro-loss, `step()` once at the end.

Two things must come out *identical* to the full batch: the accumulated `.grad` before the step, and the final weight. Find the scaling that makes it so (notes §3).

**Decision you're practicing:** accumulation-as-feature — the `loss / k` scaling and the placement of `zero_grad`/`step` around the micro-batch loop.

In [ ]:
inputs_full  = torch.tensor([1.0, 2.0, 3.0, 4.0])
targets_full = torch.tensor([2.0, 4.0, 6.0, 8.0])

def full_batch_step(lr=0.01):
    """One SGD step on the full batch. Returns (final_weight, grad_before_step)."""
    weight = torch.tensor(1.0, requires_grad=True)
    optimizer = torch.optim.SGD([weight], lr=lr)
    optimizer.zero_grad(set_to_none=True)
    loss = ((weight * inputs_full - targets_full) ** 2).mean()
    loss.backward()
    grad_before_step = weight.grad.item()
    optimizer.step()
    return weight.item(), grad_before_step

def accumulation_step(lr=0.01, num_micro_batches=2):
    """The SAME single step, but built from micro-batches of 2. Must match full_batch_step exactly."""
    # TODO: split the data (tensor.split!), scale each micro-loss, backward each, step ONCE
    pass

**Verification**

In [ ]:
# --- Verification: Exercise 4 ---
full_weight, full_grad = full_batch_step()
result = accumulation_step()
assert result is not None, "fill in the stub above first"
accum_weight, accum_grad = result

# hand-computed reference (notes §3 dry-run): per-sample grads -2, -8, -18, -32 → mean = -15
assert abs(full_grad - (-15.0)) < 1e-5, "sanity: full-batch grad should be -15"
print(f"grad   — full batch: {full_grad:+.4f}   accumulated: {accum_grad:+.4f}")
print(f"weight — full batch: {full_weight:+.6f}   accumulated: {accum_weight:+.6f}")
assert abs(accum_grad - full_grad) < 1e-5, \
    f"grads differ — got {accum_grad}, want {full_grad}. Forgot the /k? (that gives -30: a doubled LR)"
assert abs(accum_weight - full_weight) < 1e-7, "final weights differ — did step() run more than once?"
print("accumulated micro-batches reproduce the full-batch step exactly ✓")
print("Exercise 4 passed ✓")

## Exercise 5 — Pick the Right Stopper

Three scenarios, three different gradient-stoppers (notes §4's decision table). Implement each function with the *scope-appropriate* tool:

1. `build_frozen_embedding_classifier` — fine-tuning setup: the embedding must be **permanently frozen**, the classifier trainable
2. `eval_loss` — compute a loss **without building any graph** (it will be called in a metrics loop thousands of times)
3. `features_for_model_b` — feed model A's output to model B so that backward **never reaches A** (A keeps training separately on its own loss)

**Decision you're practicing:** choosing by scope — parameter-permanent vs code-block vs single-tensor.

In [ ]:
def build_frozen_embedding_classifier(vocab_size=10, embed_dim=4, num_classes=2):
    """Return (embedding, classifier) where ONLY the classifier can ever train."""
    embedding = nn.Embedding(vocab_size, embed_dim)
    classifier = nn.Linear(embed_dim, num_classes)
    # TODO: freeze the embedding — permanently, at the parameter level
    return embedding, classifier

def eval_loss(model, inputs, targets):
    """Return the CE loss WITHOUT recording any autograd graph."""
    # TODO: wrap in the right context
    pass

def features_for_model_b(model_a, inputs):
    """Return model_a's features such that a later backward CANNOT flow into model_a."""
    # TODO: cut the single tensor out of the graph
    pass

**Verification**

In [ ]:
# --- Verification: Exercise 5 ---
# (1) frozen embedding: grads never appear, weights never move
embedding, classifier = build_frozen_embedding_classifier()
weights_before = embedding.weight.clone()
optimizer = torch.optim.SGD(list(embedding.parameters()) + list(classifier.parameters()), lr=0.5)
token_batch = torch.randint(0, 10, (6, 3))                    # (batch, seq)
pooled = embedding(token_batch).mean(dim=1)                   # (batch, embed_dim)
loss = F.cross_entropy(classifier(pooled), torch.randint(0, 2, (6,)))
loss.backward()
optimizer.step()
assert embedding.weight.grad is None, "frozen embedding must never receive a gradient"
assert classifier.weight.grad is not None, "the classifier must still train!"
assert torch.equal(embedding.weight, weights_before), "frozen weights moved after optimizer.step()!"
print("(1) embedding frozen, classifier training ✓")

# (2) graph-free eval loss
eval_model = nn.Linear(4, 2)
metric = eval_loss(eval_model, torch.randn(8, 4), torch.randint(0, 2, (8,)))
assert metric is not None, "eval_loss not implemented yet"
assert metric.grad_fn is None and not metric.requires_grad, \
    "eval_loss built a graph — wrap the forward in no_grad/inference_mode"
print(f"(2) eval loss = {metric.item():.4f}, grad_fn = None ✓")

# (3) detached hand-off: backward reaches B, never A
model_a, model_b = nn.Linear(4, 4), nn.Linear(4, 2)
handoff = features_for_model_b(model_a, torch.randn(5, 4))
assert handoff is not None, "features_for_model_b not implemented yet"
model_b(handoff).sum().backward()
assert all(p.grad is None for p in model_a.parameters()), "backward leaked into model A!"
assert all(p.grad is not None for p in model_b.parameters()), "model B got no gradients"
print("(3) B trains on A's features, A untouched ✓")
print("Exercise 5 passed ✓")

## Exercise 6 — Fix the In-Place Crash

`ResidualBlockBuggy` below crashes on backward: `torch.sigmoid`'s backward **needs its own output** ($\sigma' = \sigma(1-\sigma)$), but `activated += inputs` overwrites that output in place — the version counter ticks, and backward refuses.

Reproduce the crash (pre-written), then implement `ResidualBlockFixed` with the out-of-place spelling and confirm the gradients match an independent reference.

**Decision you're practicing:** when in-place is safe (untracked tensors) vs fatal (tensors a backward formula saved) — and how to read the version-counter error.

In [ ]:
class ResidualBlockBuggy(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(3, 3)

    def forward(self, inputs):
        activated = torch.sigmoid(self.linear(inputs))   # SigmoidBackward SAVES this output
        activated += inputs                              # in-place write → version counter ticks!
        return activated.sum()

torch.manual_seed(0)
buggy_block = ResidualBlockBuggy()
try:
    buggy_block(torch.randn(2, 3)).backward()
except RuntimeError as err:
    print(f"CRASH (as designed):\n  {err}")

# TODO: the same block, out-of-place, so backward works
class ResidualBlockFixed(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(3, 3)

    def forward(self, inputs):
        # TODO
        pass

**Verification**

In [ ]:
# --- Verification: Exercise 6 ---
# the buggy block must still crash
torch.manual_seed(0)
crashed = False
try:
    ResidualBlockBuggy()(torch.randn(2, 3)).backward()
except RuntimeError as err:
    crashed = True
    assert "inplace" in str(err), f"unexpected error: {err}"
assert crashed, "the buggy block should raise the version-counter error"

# the fixed block must backward cleanly...
torch.manual_seed(0)
fixed_block = ResidualBlockFixed()
test_inputs = torch.randn(2, 3)
loss = fixed_block(test_inputs)
assert loss is not None, "implement ResidualBlockFixed.forward first"
loss.backward()
assert fixed_block.linear.weight.grad is not None

# ...and produce the same gradients as an independent out-of-place reference
reference_linear = nn.Linear(3, 3)
reference_linear.load_state_dict(fixed_block.linear.state_dict())
reference_loss = (torch.sigmoid(reference_linear(test_inputs)) + test_inputs).sum()
reference_loss.backward()
assert torch.allclose(fixed_block.linear.weight.grad, reference_linear.weight.grad, atol=1e-6), \
    "gradients differ from the reference — did the forward math change?"
assert torch.allclose(fixed_block.linear.bias.grad, reference_linear.bias.grad, atol=1e-6)
print("fixed block backward runs and matches the reference gradients ✓")
print("Exercise 6 passed ✓")

## Exercise 7 — The Loss-Logging Leak

`leaky_training` below stores the raw `loss` tensor every step. Each stored loss carries its `grad_fn` — which pins that step's **entire graph** (all saved activations) in memory, forever. On this toy problem you won't OOM; on a transformer you will, mid-epoch.

Write `clean_training`: identical training, but the history must hold **plain Python floats**.

**Decision you're practicing:** anything that outlives the step gets `.item()`ed (scalars) or `.detach()`ed (tensors).

In [ ]:
def leaky_training(num_steps=30):
    weight = torch.tensor(0.0, requires_grad=True)
    optimizer = torch.optim.SGD([weight], lr=0.05)
    loss_history = []
    for _ in range(num_steps):
        optimizer.zero_grad(set_to_none=True)
        loss = (weight * 2.0 - 4.0) ** 2
        loss.backward()
        optimizer.step()
        loss_history.append(loss)          # ← the leak: every entry pins its step's whole graph
    return weight.item(), loss_history

leaky_weight, leaky_history = leaky_training()
print(f"leaky history entry type: {type(leaky_history[0])}, grad_fn: {type(leaky_history[0].grad_fn).__name__}")
print("→ 30 stored graphs. On a real model: OOM.")

# TODO: same training, leak-free history of floats
def clean_training(num_steps=30):
    pass

**Verification**

In [ ]:
# --- Verification: Exercise 7 ---
result = clean_training()
assert result is not None, "fill in the stub above first"
clean_weight, clean_history = result

assert len(clean_history) == 30
assert all(isinstance(value, float) for value in clean_history), \
    "every logged value must be a plain Python float (.item()) — tensors pin their graphs"
assert clean_history[-1] < clean_history[0], "training broke — loss did not decrease"
assert clean_history[-1] < 1e-6, f"should converge to ~0 on this toy problem, got {clean_history[-1]}"
assert abs(clean_weight - 2.0) < 1e-3, f"optimum is w=2.0, got {clean_weight}"

# sanity contrast: the leaky version really does store graph-carrying tensors
assert isinstance(leaky_history[0], torch.Tensor) and leaky_history[0].grad_fn is not None
print(f"clean training converged (w = {clean_weight:.5f}) with a float-only history ✓")
print("Exercise 7 passed ✓")

---
## Done!

Compare your work against `solutions/ch03-autograd-mental-model-solution.ipynb`.

That completes the **language block** (ch01–ch03): tensors, the ops between them, and the gradient machinery underneath. Next up is the **model block** — **ch04 — Module Patterns**: how `nn.Module` packages parameters, buffers, and submodules, and when to reach for `Sequential` vs `ModuleList` vs `ModuleDict`.